# 04 — Evaluasi

**Peran:** R2 (AI Model Engineer)

Notebook ini melaporkan performa model akhir: terhadap target resmi, terhadap baseline pembanding, dan — yang sama pentingnya — di mana model tidak bisa diandalkan.

**Seluruh angka berasal dari split `test`**, bagian data yang tidak pernah dipakai untuk mengambil keputusan apa pun selama pengembangan. Split `val` dipakai berkali-kali untuk memilih arsitektur dan hiperparameter, sehingga angkanya sudah condong optimistis dan tidak layak jadi laporan akhir.

In [ ]:
import json
import sys

sys.path.insert(0, "../..")

metrics = json.load(open("../reports/metrics.json", encoding="utf-8"))
labels = json.load(open("../reports/labels.json", encoding="utf-8"))

print(f"Model     : {metrics['model']}")
print(f"Parameter : {metrics['parameter']:,}")
print(f"Dataset   : {metrics['dataset']}\n")

for nama, s in metrics["vs_target"].items():
    print(f"  {nama:22s} {s['nilai']:8.3f}   target {s['target']:<7} {'TERCAPAI' if s['lolos'] else 'belum'}")

## 1. Terhadap target resmi

| Head | Metrik | Hasil | Target | Status |
|---|---|---|---|---|
| Forecast suhu | MAE @ t+30 | **0,202 °C** | < 0,8 °C | tercapai |
| Mode kegagalan | Macro F1 | 0,573 | > 0,80 | belum |
| Time-to-Breach | MAE (TTB ≤ 30 menit) | **7,60 menit** | < 8 menit | tercapai |
| Deteksi anomali | PR-AUC | 0,691 | > 0,85 | belum |

**Dua dari empat tercapai.** Forecast melampaui targetnya dengan selisih besar — hampir empat kali lebih akurat daripada yang disyaratkan.

Angka TTB di tabel `metrics.json` (52,98 menit) adalah rata-rata seluruh rentang. Angka itu menyesatkan bila dibaca sendirian; penjelasannya di bagian 4.

## 2. Mengapa perlu deep learning — perbandingan baseline

Pertanyaan itu dijawab dengan eksperimen, bukan klaim. Tiga baseline diuji pada data yang sama. Karena model pohon tidak dapat memakan data berurutan, tiap jendela 60×12 diringkas menjadi 72 fitur agregat.

| Metrik | GRU fusion | XGBoost | Regresi linear | Isolation Forest |
|---|---|---|---|---|
| Forecast t+30 (°C) | 0,202 | **0,189** | 0,338 | — |
| Macro F1 | 0,573 | **0,664** | — | — |
| Akurasi | 0,795 | **0,871** | — | — |
| PR-AUC anomali | 0,691 | **0,753** | — | 0,371 |
| TTB ≤ 30 menit | 17,9 | **7,6** | — | — |

**XGBoost mengungguli GRU pada seluruh metrik.** Temuan ini dilaporkan apa adanya.

Konsekuensinya diambil: **Time-to-Breach dipindahkan ke XGBoost**, karena di situ selisihnya terbesar dan menentukan — 17,9 vs 7,6 menit, satu-satunya perpindahan yang mengubah status target dari gagal menjadi tercapai.

GRU tetap menangani forecast dan klasifikasi dengan pertimbangan: selisih forecast tipis dan keduanya jauh melampaui target; pada klasifikasi kedua model sama-sama belum mencapai 0,80 sehingga penggantian tidak mengubah papan skor; dan satu model GRU melayani dua keluaran sekaligus dalam 169 KB.

## 3. Akurasi per kelas kegagalan

In [ ]:
from IPython.display import Image, display

display(Image(filename="../reports/confusion_matrix.png"))

| Kelas | Recall | Sampel |
|---|---|---|
| A3 — kegagalan reefer total | **100,0 %** | 379 |
| A0 — sehat | 90,0 % | 12.546 |
| `masalah_sensor` (A5+A6) | 61,4 % | 872 |
| A1 — pintu terbuka lama | 53,0 % | 321 |
| A7 — kejut suhu ambien | 43,9 % | 642 |
| A8 — prapendinginan buruk | 27,0 % | 651 |
| `degradasi_bertahap` (A2+A4) | **8,3 %** | 833 |

Pola ini konsisten dengan fisikanya. **A3** menghasilkan kenaikan suhu monoton yang khas dan selalu tertangkap. **`degradasi_bertahap`** justru yang tersulit: perubahannya sangat lambat sehingga dalam jendela 60 menit nyaris tidak terbedakan dari kondisi sehat.

**Ini keterbatasan yang harus dinyatakan terbuka.** Justru mode itulah yang paling berbahaya di dunia nyata — kompresor melemah dan kebocoran refrigeran juga luput dari pengamatan manusia. Sistem **tidak boleh dipromosikan** sebagai pendeteksi kegagalan bertahap.

## 4. Time-to-Breach: akurat hanya untuk jangka pendek

In [ ]:
display(Image(filename="../reports/ttb_by_horizon.png"))

print("Model TTB produksi (XGBoost):")
for k, v in labels["model_ttb_terpisah"]["akurasi_test"].items():
    print(f"  {k:18s} {v:6.2f} menit")

| TTB sebenarnya | MAE |
|---|---|
| ≤ 10 menit | **3,46 menit** |
| ≤ 30 menit | **7,60 menit** |
| keseluruhan | 53,45 menit |

Angka tunggal 53 menit menyembunyikan hal terpenting: **model akurat persis di rentang yang menentukan keputusan.** Ketika breach masih 10 menit lagi — saat pengemudi masih sempat menepi atau memanggil bantuan — perkiraannya meleset kurang dari 3,5 menit.

Sebaliknya, memperkirakan kejadian lima jam ke depan dari jendela 60 menit berada di luar jangkauan informasi yang tersedia. Ini keterbatasan fisik, bukan kekurangan implementasi.

**Implikasi untuk antarmuka:** tampilkan angka TTB hanya bila di bawah ~30 menit. Di atas itu, tampilkan status risiko tanpa angka spesifik agar tidak memberi kesan presisi yang tidak dimiliki model.

## 5. Peringatan penerapan: keluaran TTB tidak terdefinisi saat kondisi sehat

Head TTB dilatih **hanya** pada jendela yang benar-benar menuju breach; jendela sehat dikeluarkan dari perhitungan loss. Akibatnya model tetap mengeluarkan angka untuk kondisi sehat, dan angka itu tidak bermakna.

Sel berikut membuktikannya pada 2.000 jendela sehat dari split test.

In [ ]:
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession("../reports/coldtrack_ttb.onnx")
d = np.load("../../data/processed/windows/windows_test.npz")

sehat = d["y_ttb"] == 999          # ground truth: tidak akan pernah breach
X = d["X"][sehat][:2000].astype(np.float32)
ttb = np.ravel(sess.run(None, {"window": X})[0])

print("Keluaran model untuk jendela SEHAT:")
print(f"  min {ttb.min():.1f}  |  median {np.median(ttb):.1f}  |  maks {ttb.max():.1f} menit")
print(f"  yang bernilai >= 500 (yang mungkin dikira 'aman'): {(ttb >= 500).mean() * 100:.2f}%")
print("\nAngka-angka ini TIDAK BERMAKNA -- model tidak pernah dilatih pada kondisi sehat.")
print("Tidak ada nilai sentinel besar yang bisa dipakai backend sebagai penanda aman.")
print("Backend WAJIB menyembunyikan TTB bila failure_prob menunjuk kelas A0.")

## 6. Ekspor produksi

| | `coldtrack.onnx` | `coldtrack_ttb.onnx` |
|---|---|---|
| Isi | forecast + klasifikasi | Time-to-Breach |
| Ukuran | 169 KB | 946 KB |
| Latensi CPU | 1,1 ms | 0,09 ms |
| Kesetaraan numerik | ~1e-6 (syarat < 1e-4) | lihat catatan |

**Kontrak masukan kedua model identik** — `window` berbentuk `[batch, 60, 12]` float32, nilai mentah. Tensor yang sama dapat dikirim ke keduanya; backend tidak perlu prapemrosesan tambahan, karena normalisasi dan perhitungan agregat dibungkus di dalam grafik ONNX masing-masing.

**Catatan kesetaraan model TTB.** Perhitungan agregat dibungkus ke dalam grafik, sehingga pembulatan float32 kadang membuat pohon keputusan mengambil cabang berbeda. Sekitar 67% prediksi berbeda tipis dari XGBoost yang dipanggil langsung, tetapi dampaknya pada MAE keseluruhan di bawah 0,5 menit (53,45 vs 53,76). Kesetaraan ketat 1e-4 karena itu tidak dapat diklaim untuk model ini — dinyatakan terbuka alih-alih dilewatkan.

In [ ]:
import time

for berkas in ["coldtrack.onnx", "coldtrack_ttb.onnx"]:
    s = ort.InferenceSession(f"../reports/{berkas}")
    x = d["X"][:1].astype(np.float32)
    s.run(None, {"window": x})                      # pemanasan
    t0 = time.perf_counter()
    for _ in range(200):
        s.run(None, {"window": x})
    print(f"{berkas:22s} {(time.perf_counter()-t0)/200*1000:5.2f} ms   "
          f"(batas 300 ms)")

## 7. Ringkasan

**Yang berhasil.** Forecast suhu sangat akurat (0,202 °C, target 0,8). Time-to-Breach memenuhi target pada rentang yang menentukan (7,6 menit, target 8). Kegagalan reefer total terdeteksi sempurna. Kedua model ringan dan jauh di bawah batas latensi.

**Yang belum.** Klasifikasi mode kegagalan (0,573) dan deteksi anomali (0,691) belum mencapai target. Mode degradasi bertahap hampir tidak terdeteksi.

**Yang dilaporkan jujur.** XGBoost mengungguli GRU pada seluruh metrik; prapelatihan tidak memberi manfaat dan justru merugikan; model belum pernah divalidasi terhadap perjalanan truk sungguhan.

Keterbatasan lengkap ada di `docs/model_card.md`.